<a href="https://colab.research.google.com/github/aronwilbert/COMP8420-Group-L-Healthcare/blob/main/COMP8420_Group_L_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --force-reinstall numpy pandas datasets
!python -m spacy download en_core_web_md

  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)


In [2]:
from datasets import load_dataset, DatasetDict
import pandas as pd
import sys
import re
import pandas as pd
from textblob import TextBlob
from sklearn.metrics.pairwise import cosine_similarity
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from IPython.display import display_html

# Ensure scispacy packages are present
try:
    import scispacy
    import spacy
    if not spacy.util.is_package("en_ner_bc5cdr_md"):
        raise ImportError
except ImportError:
    print("⏳ Downloading biomedical language weights...")
    !{sys.executable} -m pip install -q scispacy
    !{sys.executable} -m pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
    import scispacy
    import spacy

In [3]:
# 1. Load the raw master dataset
master_dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
print(f"📊 Initial Raw Master Dataset Samples: {len(master_dataset)}")

# --- VISUAL 1: DISPLAY RAW DATASET BEFORE CLEANING ---
print("\n--- 🛑 RAW DATASET PREVIEW (BEFORE CLEANING) ---")
df_raw_preview = pd.DataFrame(master_dataset.select(range(5)))
display(df_raw_preview)

# 2. Filter out rows where 'input' or 'output' are missing or completely empty
clean_master = master_dataset.filter(
    lambda x: x["input"] is not None and x["output"] is not None and
              str(x["input"]).strip() != "" and str(x["output"]).strip() != ""
)
print(f"\n🧹 Cleaned Master Dataset Samples (After NA/Blank Removal): {len(clean_master)}")
print(f"❌ Total Defective Rows Removed: {len(master_dataset) - len(clean_master)}")

# 3. Securely isolate EXACTLY 5,000 clean samples for the project
dataset = clean_master.select(range(5000))
print(f"✅ Final Target Samples Isolated for Training/Analysis: {len(dataset)}")

# --- VISUAL 2: DISPLAY FINAL DATASET AFTER CLEANING ---
df_clean_final = pd.DataFrame({
    'input': [row['input'] for row in dataset],
    'output': [row['output'] for row in dataset]
})

print("\n--- 📋 VERIFIED CLEAN DATASET SNAPSHOT (AFTER CLEANING) ---")
display(df_clean_final.head(5))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


📊 Initial Raw Master Dataset Samples: 112165

--- 🛑 RAW DATASET PREVIEW (BEFORE CLEANING) ---


,conversations,input,output
0,"[{'from': 'human', 'value': 'I woke up this mo...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"[{'from': 'human', 'value': 'My baby has been ...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"[{'from': 'human', 'value': 'Hello, My husband...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"[{'from': 'human', 'value': 'lump under left n...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"[{'from': 'human', 'value': 'I have a 5 month ...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...



🧹 Cleaned Master Dataset Samples (After NA/Blank Removal): 112156
❌ Total Defective Rows Removed: 9
✅ Final Target Samples Isolated for Training/Analysis: 5000

--- 📋 VERIFIED CLEAN DATASET SNAPSHOT (AFTER CLEANING) ---


,input,output
0,I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [4]:
# Ensure master data list is isolated
raw_documents = df_clean_final['input'].astype(str).tolist()

# =====================================================================
# 📋 STEP 1: PRE-CLEANING BASELINE ANALYSIS (STOPWORDS REMOVED FOR MATH ONLY)
# =====================================================================
print("=== 📋 STEP 1: PRE-CLEANING BASELINE PROFILE ===")

pre_tfidf = TfidfVectorizer(stop_words='english', max_features=10)
pre_tfidf_matrix = pre_tfidf.fit_transform(raw_documents)
df_pre_tfidf = pd.DataFrame({'Pre-Clean Word': pre_tfidf.get_feature_names_out(), 'TF-IDF Score': pre_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

pre_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10)
pre_count_matrix = pre_count.fit_transform(raw_documents)
df_pre_ngrams = pd.DataFrame({'Pre-Clean Phrase': pre_count.get_feature_names_out(), 'Frequency Count': pre_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

html_pre = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Word Importance Baseline</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Multi-Word Phrase Baseline</h3>
        {}
    </div>
</div>
""".format(df_pre_tfidf.to_html(index=False), df_pre_ngrams.to_html(index=False))
display_html(html_pre, raw=True)


# =====================================================================
# 🧼 STEP 2: CAREFULLY ORDERED DATA CLEANING PASS
# =====================================================================
print("\n" + "="*80 + "\n=== 🧼 STEP 2: EXECUTING CAREFULLY ORDERED PIPELINE ===")

def ultimate_medical_cleaner(text):
    if not isinstance(text, str): return ""

    # 1. Expand contractions to safeguard downstream word structures
    contractions = {
        r"can't": "cannot", r"don't": "do not", r"it's": "it is", r"i'm": "i am",
        r"doesn't": "does not", r"didn't": "did not", r"he's": "he is", r"she's": "she is"
    }
    for contraction, expansion in contractions.items():
        text = re.sub(contraction, expansion, text, flags=re.IGNORECASE)

    # 2. Add spaces after jammed conversational punctuation
    text = re.sub(r'([,;\.\!\?])([A-Za-z])', r'\1 \2', text)

    # 3. Strip greetings and sign-offs at boundaries
    text = re.sub(r'^\b(hi+|hello+|hey|dear)\b\s*\b(doc|doctor)?\b[,\s\.]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(thank\s*you|thanks)\s*\b(doc|doctor)?\b[!\.]*$', '', text, flags=re.IGNORECASE)

    # 4. Extract and mask complex age features
    text = re.sub(r'\b\d+\s*(years?|yrs?|months?|mos?|mths?)\s*(old)?\b', '[REDACTED_AGE]', text, flags=re.IGNORECASE)

    # 5. Targeted Punctuation Swap: Replace punctuation with spaces, KEEPING hyphens (-) and brackets ([])
    text = re.sub(r'[^\w\s\-\[\]]', ' ', text)

    # 6. Compress formatting layouts into clear singular spaces
    return re.sub(r'\s+', ' ', text).strip()

# Apply to your dataset
df_clean_final['clean_input'] = df_clean_final['input'].apply(ultimate_medical_cleaner)
clean_documents = df_clean_final['clean_input'].tolist()

print("✅ Master Dataset Cleaned Successfully.")


# =====================================================================
# ✅ STEP 3: POST-CLEANING VERIFICATION ANALYSIS
# =====================================================================
print("\n" + "="*80 + "\n=== ✅ STEP 3: POST-CLEANING VERIFICATION PROFILE ===")

post_tfidf = TfidfVectorizer(stop_words='english', max_features=10, token_pattern=r'(?u)\b\w+\b|\[\w+\]')
post_tfidf_matrix = post_tfidf.fit_transform(clean_documents)
df_post_tfidf = pd.DataFrame({'Cleaned Term': post_tfidf.get_feature_names_out(), 'TF-IDF Score': post_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

post_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10, token_pattern=r'(?u)\b\w+\b|\[\w+\]')
post_count_matrix = post_count.fit_transform(clean_documents)
df_post_ngrams = pd.DataFrame({'Cleaned Phrase': post_count.get_feature_names_out(), 'Frequency Count': post_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

html_post = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Cleaned Word Importance</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Cleaned Multi-Word Phrases</h3>
        {}
    </div>
</div>
""".format(df_post_tfidf.to_html(index=False), df_post_ngrams.to_html(index=False))
display_html(html_post, raw=True)

display(df_clean_final[['input', 'clean_input']].head(3))

=== 📋 STEP 1: PRE-CLEANING BASELINE PROFILE ===


Pre-Clean Word,TF-IDF Score
pain,0.188483
hi,0.136012
old,0.131301
like,0.129531
doctor,0.123912
days,0.119800
years,0.119387
just,0.116030
ago,0.098343
right,0.097079



=== 🧼 STEP 2: EXECUTING CAREFULLY ORDERED PIPELINE ===
✅ Master Dataset Cleaned Successfully.

=== ✅ STEP 3: POST-CLEANING VERIFICATION PROFILE ===


Cleaned Term,TF-IDF Score
[redacted_age],0.230689
pain,0.179547
like,0.127874
days,0.119208
just,0.114804
doctor,0.106497
ago,0.096186
right,0.095093
time,0.095007
left,0.081199


,input,clean_input
0,I woke up this morning feeling the whole room ...,I woke up this morning feeling the whole room ...
1,My baby has been pooing 5-6 times a day for a ...,My baby has been pooing 5-6 times a day for a ...
2,"Hello, My husband is taking Oxycodone due to a...",My husband is taking Oxycodone due to a broken...


In [5]:
import re

# Pick an excellent sample row that contains contractions, mashed punctuation, greetings, and ages
sample_raw_text = df_clean_final['input'].iloc[3]

print("================================================================================")
print("🔬 STEP-BY-STEP DATA ENGINEERING TRACE PIPELINE")
print("================================================================================\n")

print(f"🔴 [0. ORIGINAL RAW TEXT]:\n{sample_raw_text}\n")
print("-" * 80)

# --- STEP 1: CONTRACTION EXPANSION ---
text_step1 = sample_raw_text
contractions = {
    r"can't": "cannot", r"don't": "do not", r"it's": "it is", r"i'm": "i am",
    r"doesn't": "does not", r"didn't": "did not", r"he's": "he is", r"she's": "she is"
}
for contraction, expansion in contractions.items():
    text_step1 = re.sub(contraction, expansion, text_step1, flags=re.IGNORECASE)

print(f"🔄 [1. AFTER CONTRACTION EXPANSION] (e.g., don't -> do not):\n{text_step1}\n")
print("-" * 80)

# --- STEP 2: PUNCTUATION SPACING ---
text_step2 = re.sub(r'([,;\.\!\?])([A-Za-z])', r'\1 \2', text_step1)

print(f"🔄 [2. AFTER PUNCTUATION SPACING ADJUSTMENT] (e.g., Hi,I -> Hi, I):\n{text_step2}\n")
print("-" * 80)

# --- STEP 3: GREETING & SIGN-OFF REMOVAL ---
text_step3 = re.sub(r'^\b(hi+|hello+|hey|dear)\b\s*\b(doc|doctor)?\b[,\s\.]*', '', text_step2, flags=re.IGNORECASE)
text_step3 = re.sub(r'\b(thank\s*you|thanks)\s*\b(doc|doctor)?\b[!\.]*$', '', text_step3, flags=re.IGNORECASE)

print(f"🔄 [3. AFTER RULE-BASED GREETING & SIGN-OFF REMOVAL]:\n{text_step3}\n")
print("-" * 80)

# --- STEP 4: AGE ANONYMIZATION ---
text_step4 = re.sub(r'\b\d+\s*(years?|yrs?|months?|mos?|mths?)\s*(old)?\b', '[REDACTED_AGE]', text_step3, flags=re.IGNORECASE)

print(f"🔄 [4. AFTER PATTERN-BASED AGE REDACTION]:\n{text_step4}\n")
print("-" * 80)

# --- STEP 5: TARGETED PUNCTUATION FILTER ---
text_step5 = re.sub(r'[^\w\s\-\[\]]', ' ', text_step4)

print(f"🔄 [5. AFTER PUNCTUATION NORMALIZATION] (Keeps hyphens and brackets):\n{text_step5}\n")
print("-" * 80)

# --- STEP 6: WHITESPACE COMPRESSION ---
text_step6 = re.sub(r'\s+', ' ', text_step5).strip()

print(f"🟢 [6. FINAL CLEAN OUTPUT SENTENCE FOR CHATBOT]:\n{text_step6}\n")
print("================================================================================")

🔬 STEP-BY-STEP DATA ENGINEERING TRACE PIPELINE

🔴 [0. ORIGINAL RAW TEXT]:
lump under left nipple and stomach pain (male) Hi,I have recently noticed a few weeks ago a lump under my nipple, it hurts to touch and is about the size of a quarter. Also I have bern experiencing stomach pains that prevent me from eating. I immediatly feel full and have extreme pain. Please help

--------------------------------------------------------------------------------
🔄 [1. AFTER CONTRACTION EXPANSION] (e.g., don't -> do not):
lump under left nipple and stomach pain (male) Hi,I have recently noticed a few weeks ago a lump under my nipple, it hurts to touch and is about the size of a quarter. Also I have bern experiencing stomach pains that prevent me from eating. I immediatly feel full and have extreme pain. Please help

--------------------------------------------------------------------------------
🔄 [2. AFTER PUNCTUATION SPACING ADJUSTMENT] (e.g., Hi,I -> Hi, I):
lump under left nipple and stomach pa

In [7]:
# Load execution models
nlp = spacy.load("en_core_web_md")
nlp_medical = spacy.load("en_ner_bc5cdr_md")

print("================================================================================")
print("🧬 ADVANCED/BASIC HYBRID TECHNIQUE: CLINICAL NAMED ENTITY RECOGNITION")
print("================================================================================")
patient_text = df_clean_final['clean_input'].iloc[2]
doctor_text = df_clean_final['output'].iloc[2]

doc_patient_med = nlp_medical(patient_text)
doc_doctor_med = nlp_medical(doctor_text)

print(f"\n🔴 Cleaned Patient Input:\n\"{patient_text}\"")
print("\nDetected Clinical Entities:")
if not doc_patient_med.ents:
    print("   No clinical entities detected.")
else:
    for ent in doc_patient_med.ents:
        print(f"   👉 '{ent.text}' ➔ Clinical Type: {ent.label_}")

print(f"\n🟢 Raw Doctor Output:\n\"{doctor_text[:250]}...\"")
print("\nDetected Clinical Entities:")
if not doc_doctor_med.ents:
    print("   No clinical entities detected.")
else:
    for ent in doc_doctor_med.ents:
        print(f"   👉 '{ent.text}' ➔ Clinical Type: {ent.label_}")

print("\n" + "="*80 + "\n")

print("================================================================================")
print("🧠 BASIC TECHNIQUE: TEXT SIMILARITY (SENTENCE EMBEDDINGS & COSINE)")
print("================================================================================")
doc_patient = nlp(patient_text)
doc_doctor = nlp(doctor_text)
unrelated_doctor_text = df_clean_final['output'].iloc[0]
doc_unrelated = nlp(unrelated_doctor_text)

vec_patient = doc_patient.vector.reshape(1, -1)
vec_correct_doc = doc_doctor.vector.reshape(1, -1)
vec_unrelated_doc = doc_unrelated.vector.reshape(1, -1)

sim_correct = cosine_similarity(vec_patient, vec_correct_doc)[0][0]
sim_unrelated = cosine_similarity(vec_patient, vec_unrelated_doc)[0][0]

print(f"✅ Cosine Similarity (Patient vs. Correct Doctor Match)  : {sim_correct:.4f}")
print(f"❌ Cosine Similarity (Patient vs. Unrelated Doctor Match): {sim_unrelated:.4f}\n")

print("================================================================================")
print("🎭 BASIC TECHNIQUE: SENTIMENT ANALYSIS FOR PATIENT SATISFACTION")
print("================================================================================")
painful_patient_text = df_clean_final['clean_input'].iloc[3]
blob_analysis = TextBlob(painful_patient_text)
sentiment_score = blob_analysis.sentiment.polarity

print(f"📄 Analyzing Patient Query:\n\"{painful_patient_text}\"")
print(f"\n🎭 Calculated Sentiment Polarity Score: {sentiment_score:.4f}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject